# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook demonstrates loading, processing, and analyzing the FAIR^2 dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset is defined by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset package and records from the Croissant schema using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for clarity in the notebook

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n\nPublished: {metadata.date_published if hasattr(metadata, 'date_published') else metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")

## 2. Data Overview
Explore available record sets, fields, and their `@id`s. This step utilizes `dataset.record_sets` to list all record sets and their details.

In [ ]:
# List all available record sets by their @id and name
print("Available record sets in this dataset:")
for record_set in dataset.record_sets:
    print(f"  @id: {record_set['@id']}")
    print(f"    name: {record_set.get('name', '(no name)')}")
    if 'field' in record_set:
        print("    Fields:")
        for field in record_set['field']:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
            print(f"      - {field_id}")
    print()

# For deeper info, view the first 1-2 record sets if present
if len(dataset.record_sets) > 0:
    print("Example fields for the first record set:")
    rs = dataset.record_sets[0]
    display_fields = rs.get('field', [])
    print(f"Record set @id: {rs['@id']}")
    for f in display_fields:
        if isinstance(f, dict):
            fid = f['@id']
            fname = f.get('name', '(no name)')
        else:
            fid, fname = str(f), ''
        print(f"  - Field @id: {fid}  name: {fname}")

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

*Replace `<record_set_id>` below with the actual `@id` of a record set you want to inspect. Multiple record sets can be accessed using their `@id`s.*

In [ ]:
# Collect all record set @id values for iteration
available_record_sets = [rs['@id'] for rs in dataset.record_sets]
print("Record set @id list:", available_record_sets)

# Define record sets to extract (replace with actual desired id(s))
record_sets = available_record_sets
dataframes = {}

for record_set_id in record_sets:
    print(f"\nLoading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print("  No records found.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded {df.shape[0]} rows, columns: {df.columns.tolist()}")

# Example: show first few rows of the first record set loaded
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"\nFirst 5 rows of RecordSet {first_rs_id}:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, grouping, and prepare the data for further analysis.

*In this section, select numeric and grouping field `@id`s as appropriate from the columns loaded above.*

In [ ]:
# For demonstration, use the first available record set with numeric fields
selected_rs_id = None
numeric_field_id = None
group_field_id = None
df = None

for rs_id, frame in dataframes.items():
    numeric_cols = frame.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        selected_rs_id = rs_id
        numeric_field_id = numeric_cols[0]  # use first numeric
        # Example: pick a non-numeric for grouping if available
        group_candidates = [c for c in frame.columns if c != numeric_field_id]
        group_field_id = group_candidates[0] if len(group_candidates) else None
        df = frame
        break

print(f"Analyzing record set: {selected_rs_id}")
print(f"Numeric field selected: {numeric_field_id}")
print(f"Group field selected: {group_field_id}")

# If numeric field found, proceed with EDA
if df is not None and numeric_field_id is not None:
    # Example threshold: 10 (you may adjust based on your column)
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}, count: {filtered_df.shape[0]}")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field_id if available
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships of fields in the extracted data.

*Replace the visualization as needed depending on the available data.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id} in {selected_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If a group_field_id exists, show boxplot/grouped plot
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This exploratory notebook demonstrates how to:
- Load a Croissant-formatted dataset and metadata using `mlcroissant`
- Discover record sets, fields, and work with their `@id`s
- Extract records as DataFrames, filter, normalize, and group data
- Visualize numeric fields using histograms and boxplots

For further analysis, consult field descriptions and the dataset's Croissant metadata, referencing all entities by their `@id`s for reproducibility and clarity.